# Ropedia Academy — D3 · TSDF fusion & submaps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/D3.ipynb)

> **Runs TSDF fusion over 50 noisy depth frames — truncated signed distances + weighted averaging — and extracts the denoised surface as the zero-crossing.**
>
> 对 50 帧含噪深度做 TSDF 融合——截断符号距离 + 加权平均——并以零交叉提取去噪后的表面。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/D3

In [ ]:
import numpy as np
np.random.seed(0)

# TSDF fusion: average many NOISY depth frames into one clean surface.
# 1D slice along a ray for clarity; true surface at depth 1.0.
voxels = np.linspace(0, 2, 41); trunc = 0.2
tsdf = np.zeros_like(voxels); weight = np.zeros_like(voxels)

def integrate(tsdf, weight, depth):                 # one depth frame
    sdf  = np.clip((depth - voxels) / trunc, -1, 1) # TRUNCATED signed distance
    near = np.abs(depth - voxels) < trunc
    tsdf[near]   = (weight[near]*tsdf[near] + sdf[near]) / (weight[near] + 1)  # weighted avg
    weight[near] += 1
    return tsdf, weight

for _ in range(50):                                  # 50 noisy depth measurements
    tsdf, weight = integrate(tsdf, weight, 1.0 + np.random.randn()*0.05)

# Surface = zero-crossing of the fused TSDF (what Marching Cubes extracts).
zc = np.where(np.diff(np.sign(tsdf[weight > 0])))[0]
print("fused surface depth:", voxels[weight > 0][zc].round(3), "(true 1.0) -> noise averaged out")
print("truncation stores only the near-surface band; submaps tile big scenes")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/D3
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks